# EXP-21: v14 + More Models — CatBoost + Multi-Seed SVCs + SVD Topics

**Competition:** SPR 2026 — Mammography Report Classification  
**Base:** EXP-14 (0.80328 LB)  
**Melhorias:**
1. **CatBoost** como 8o modelo no Level 1 (diversidade de arquitetura)
2. **Multi-seed SVCs** (2 seeds por SVC type → 6 SVCs ao inves de 3)
3. **SVD topic features** para LGB/CatBoost (50 dims, do exp17)
4. **Dirichlet weight search** como alternativa ao meta-learner
5. Threshold estendido [6,5,4,3,1,0]
6. Dense features expandidas (26+)

**Runtime:** CPU only

In [ ]:
import os, re, hashlib, warnings, time
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.pipeline import FeatureUnion
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score, classification_report
from sklearn.decomposition import TruncatedSVD
from scipy.sparse import hstack, csr_matrix
import lightgbm as lgb

try:
    from catboost import CatBoostClassifier
except ImportError:
    !pip install -q catboost
    from catboost import CatBoostClassifier

warnings.filterwarnings('ignore')

TRAIN_PATH = '/kaggle/input/competitions/spr-2026-mammography-report-classification/train.csv'
TEST_PATH  = '/kaggle/input/competitions/spr-2026-mammography-report-classification/test.csv'

train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)

N_FOLDS = 5
N_CLASSES = 7

print(f'Train: {train.shape}, Test: {test.shape}')
print(f'Class distribution:\n{train["target"].value_counts().sort_index()}')

In [ ]:
# =============================================================================
# Cell 2: Preprocessing (igual exp14)
# =============================================================================

def clean_achados(s):
    if pd.isna(s): return ""
    s = str(s).strip().lower()
    s = s.replace("\r\n", "\n").replace("\r", "\n")
    s = re.sub(r"[ \t]+", " ", s)
    s = re.sub(r"\n{2,}", "\n", s)
    match = re.search(r'achados[:\s]*(.*?)(?:impress[aã]o|conclus[aã]o|an[aá]lise comparativa|opini[aã]o|$)', s, flags=re.DOTALL)
    if match: s = match.group(1).strip()
    s = re.sub(r'[0-9]+,[0-9]+', 'NUM', s)
    s = re.sub(r'[0-9]+', 'NUM', s)
    return s

def clean_full(s):
    if pd.isna(s): return ""
    s = str(s).strip().lower()
    s = s.replace("\r\n", "\n").replace("\r", "\n")
    s = re.sub(r"[ \t]+", " ", s)
    s = re.sub(r'[0-9]+,[0-9]+', 'NUM', s)
    s = re.sub(r'[0-9]+', 'NUM', s)
    return s

def stable_hash(s):
    return hashlib.md5(s.encode("utf-8")).hexdigest()

def extract_birads_number(text):
    match = re.search(r'(?:bi-?rads|categoria)\s*:?\s*(\d)', text)
    return int(match.group(1)) if match else -1

def extract_dense_features(df):
    feat = pd.DataFrame(index=df.index)
    text_col = df['report'].fillna('').astype(str).str.lower()
    feat['report_length']      = text_col.apply(len)
    feat['word_count']         = text_col.str.split().str.len().fillna(0).astype(int)
    feat['has_measurement']    = text_col.str.contains(r'\b(?:cm|mm|medindo)\b', regex=True).astype(int)
    feat['has_spiculation']    = text_col.str.contains(r'espiculad', regex=True).astype(int)
    feat['has_distortion']     = text_col.str.contains(r'distor[cç][aã]o arquitetural', regex=True).astype(int)
    feat['has_biopsy']         = text_col.str.contains(r'bi[oó]psia|carcinoma|resultado de cine', regex=True).astype(int)
    feat['has_nodule']         = text_col.str.contains(r'n[oó]dulo', regex=True).astype(int)
    feat['has_calcification']  = text_col.str.contains(r'calcifica[cç]', regex=True).astype(int)
    feat['has_microcalc']      = text_col.str.contains(r'microcalcifica', regex=True).astype(int)
    feat['has_asymmetry']      = text_col.str.contains(r'assimetria', regex=True).astype(int)
    feat['has_suspicious']     = text_col.str.contains(r'suspeit[oa]|malign[oa]', regex=True).astype(int)
    feat['has_benign']         = text_col.str.contains(r'benign[oa]|cisto[s]?\b', regex=True).astype(int)
    feat['has_nodulo_espic']   = text_col.str.contains(r'n[oó]dulo\s+espiculad', regex=True).astype(int)
    feat['has_irregular']      = text_col.str.contains(r'irregular', regex=True).astype(int)
    feat['has_heterogeneo']    = text_col.str.contains(r'heterog[eê]ne', regex=True).astype(int)
    feat['has_birads_explicit'] = text_col.str.contains(r'bi-?rads|categoria\s*\d', regex=True).astype(int)
    feat['birads_mentioned']    = text_col.apply(extract_birads_number)
    feat['risk_score'] = (
        feat['has_spiculation'] * 3 + feat['has_distortion'] * 3 +
        feat['has_nodulo_espic'] * 4 + feat['has_biopsy'] * 5 +
        feat['has_suspicious'] * 3 + feat['has_irregular'] * 2 +
        feat['has_microcalc'] * 2 + feat['has_asymmetry'] * 2 -
        feat['has_benign'] * 2
    )
    return feat

train["achados"] = train["report"].apply(clean_achados)
train["full"]    = train["report"].apply(clean_full)
test["achados"]  = test["report"].apply(clean_achados)
test["full"]     = test["report"].apply(clean_full)
train["group"]   = train["report"].apply(stable_hash)

dense_train = extract_dense_features(train)
dense_test  = extract_dense_features(test)
X_train_dense = csr_matrix(dense_train.values.astype(np.float32))
X_test_dense  = csr_matrix(dense_test.values.astype(np.float32))

y = train["target"].astype(int).values
groups = train["group"].values

print(f'Dense features: {X_train_dense.shape[1]}')

In [ ]:
# =============================================================================
# Cell 3: TF-IDF + SVD Topics
# =============================================================================

print("Building TF-IDF features...")

tfidf_A = FeatureUnion([
    ("word", TfidfVectorizer(ngram_range=(1, 3), min_df=3, max_df=0.95, sublinear_tf=True)),
    ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=3, max_df=0.95,
                             sublinear_tf=True, max_features=80000))
])
X_train_A = tfidf_A.fit_transform(train["achados"].values)
X_test_A  = tfidf_A.transform(test["achados"].values)

tfidf_F = FeatureUnion([
    ("word", TfidfVectorizer(ngram_range=(1, 3), min_df=3, max_df=0.95, sublinear_tf=True)),
    ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=3, max_df=0.95,
                             sublinear_tf=True, max_features=80000))
])
X_train_F = tfidf_F.fit_transform(train["full"].values)
X_test_F  = tfidf_F.transform(test["full"].values)

tfidf_F2 = FeatureUnion([
    ("word", TfidfVectorizer(ngram_range=(1, 3), min_df=3, max_df=0.95, sublinear_tf=True)),
    ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 6), min_df=3, max_df=0.95,
                             sublinear_tf=True, max_features=100000))
])
X_train_F2 = tfidf_F2.fit_transform(train["full"].values)
X_test_F2  = tfidf_F2.transform(test["full"].values)

# NOVO: SVD topic features (do exp17)
print("Computing SVD topics...")
X_all_sparse = hstack([X_train_F, X_train_A]).tocsr()
X_all_sparse_te = hstack([X_test_F, X_test_A]).tocsr()
svd = TruncatedSVD(n_components=50, random_state=42)
X_tr_svd = svd.fit_transform(X_all_sparse)
X_te_svd = svd.transform(X_all_sparse_te)
print(f"SVD explained variance: {svd.explained_variance_ratio_.sum():.3f}")

# LGB/CatBoost features: achados TF-IDF + dense + SVD topics
X_train_lgb = hstack([X_train_A, X_train_dense, csr_matrix(X_tr_svd)]).tocsr()
X_test_lgb  = hstack([X_test_A,  X_test_dense,  csr_matrix(X_te_svd)]).tocsr()

# CatBoost: dense-only (SVD + dense features)
X_tr_catboost = np.hstack([X_tr_svd, dense_train.values.astype(np.float32)])
X_te_catboost = np.hstack([X_te_svd, dense_test.values.astype(np.float32)])

# Full+dense for LR
X_train_full_dense = hstack([X_train_F, X_train_dense]).tocsr()
X_test_full_dense  = hstack([X_test_F,  X_test_dense]).tocsr()

print(f"SVC-A: {X_train_A.shape[1]}, SVC-F: {X_train_F.shape[1]}, SVC-F2: {X_train_F2.shape[1]}")
print(f"LGB: {X_train_lgb.shape[1]}, CatBoost: {X_tr_catboost.shape[1]}")

In [ ]:
# =============================================================================
# Cell 4: Level 1 — 10 Base Models (exp14's 7 + multi-seed SVCs + CatBoost)
# =============================================================================

def softmax(x):
    e = np.exp(x - x.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)

gkf = GroupKFold(n_splits=N_FOLDS)
folds = list(gkf.split(X_train_A, y, groups))

oof_probas = {}
test_probas = {}

# Multi-seed SVCs: seed 42 (original) + seed 123 (diversidade)
L1_MODELS = [
    # --- SVCs seed 42 (original exp14) ---
    ('svc_A_s42', lambda: CalibratedClassifierCV(
        LinearSVC(class_weight="balanced", random_state=42, max_iter=10000, C=1.0), cv=3, method='sigmoid'),
     X_train_A, X_test_A),
    ('svc_F_s42', lambda: CalibratedClassifierCV(
        LinearSVC(class_weight="balanced", random_state=42, max_iter=10000, C=1.0), cv=3, method='sigmoid'),
     X_train_F, X_test_F),
    ('svc_F2_s42', lambda: CalibratedClassifierCV(
        LinearSVC(class_weight="balanced", random_state=42, max_iter=10000, C=1.0), cv=3, method='sigmoid'),
     X_train_F2, X_test_F2),
    # --- SVCs seed 123 (NOVO: diversidade de seed) ---
    ('svc_A_s123', lambda: CalibratedClassifierCV(
        LinearSVC(class_weight="balanced", random_state=123, max_iter=10000, C=0.8), cv=3, method='sigmoid'),
     X_train_A, X_test_A),
    ('svc_F_s123', lambda: CalibratedClassifierCV(
        LinearSVC(class_weight="balanced", random_state=123, max_iter=10000, C=0.8), cv=3, method='sigmoid'),
     X_train_F, X_test_F),
    ('svc_F2_s123', lambda: CalibratedClassifierCV(
        LinearSVC(class_weight="balanced", random_state=123, max_iter=10000, C=1.5), cv=3, method='sigmoid'),
     X_train_F2, X_test_F2),
    # --- LGBs (original exp14 + SVD features) ---
    ('lgb1', lambda: lgb.LGBMClassifier(
        class_weight='balanced', n_estimators=300, learning_rate=0.05,
        max_depth=6, random_state=42, n_jobs=-1, verbose=-1),
     X_train_lgb, X_test_lgb),
    ('lgb2', lambda: lgb.LGBMClassifier(
        class_weight='balanced', n_estimators=500, learning_rate=0.03,
        max_depth=7, num_leaves=63, min_child_samples=20,
        subsample=0.8, colsample_bytree=0.6,
        random_state=123, n_jobs=-1, verbose=-1),
     X_train_lgb, X_test_lgb),
    # --- LR + Ridge (original exp14) ---
    ('lr', lambda: LogisticRegression(
        class_weight='balanced', C=1.0, max_iter=5000,
        solver='lbfgs', multi_class='multinomial', random_state=42, n_jobs=-1),
     X_train_full_dense, X_test_full_dense),
    ('ridge', lambda: RidgeClassifier(class_weight='balanced', alpha=1.0),
     X_train_F, X_test_F),
]

t0 = time.time()
for name, model_fn, X_tr, X_te in L1_MODELS:
    print(f"--- {name} ---")
    oof = np.zeros((len(train), N_CLASSES), dtype=np.float64)
    te_acc = np.zeros((len(test), N_CLASSES), dtype=np.float64)
    for fi, (tr_idx, va_idx) in enumerate(folds):
        m = model_fn()
        m.fit(X_tr[tr_idx], y[tr_idx])
        if hasattr(m, 'predict_proba'):
            oof[va_idx] = m.predict_proba(X_tr[va_idx])
            te_acc += m.predict_proba(X_te) / N_FOLDS
        else:
            oof[va_idx] = softmax(m.decision_function(X_tr[va_idx]))
            te_acc += softmax(m.decision_function(X_te)) / N_FOLDS
    oof_f1 = f1_score(y, np.argmax(oof, axis=1), average='macro')
    print(f"  OOF F1: {oof_f1:.4f}")
    oof_probas[name] = oof
    test_probas[name] = te_acc

# NOVO: CatBoost (dense features only — rapido e complementar)
print("--- catboost ---")
oof_cb = np.zeros((len(train), N_CLASSES), dtype=np.float64)
te_cb  = np.zeros((len(test), N_CLASSES), dtype=np.float64)
for fi, (tr_idx, va_idx) in enumerate(folds):
    m = CatBoostClassifier(iterations=500, learning_rate=0.05, depth=6, l2_leaf_reg=3,
                            auto_class_weights='Balanced', random_seed=42, verbose=0, task_type='CPU')
    m.fit(X_tr_catboost[tr_idx], y[tr_idx],
          eval_set=(X_tr_catboost[va_idx], y[va_idx]), early_stopping_rounds=50)
    oof_cb[va_idx] = m.predict_proba(X_tr_catboost[va_idx])
    te_cb += m.predict_proba(X_te_catboost) / N_FOLDS
cb_f1 = f1_score(y, np.argmax(oof_cb, axis=1), average='macro')
print(f"  OOF F1: {cb_f1:.4f}")
oof_probas['catboost'] = oof_cb
test_probas['catboost'] = te_cb

print(f"\nAll {len(oof_probas)} models trained in {time.time()-t0:.0f}s")

In [ ]:
# =============================================================================
# Cell 5: Level 2 — Meta-Learner + Dirichlet Weight Search
# =============================================================================

model_names = list(oof_probas.keys())
n_models = len(model_names)
X_meta_train = np.hstack([oof_probas[name] for name in model_names])
X_meta_test  = np.hstack([test_probas[name] for name in model_names])
print(f"Meta-features: {X_meta_train.shape[1]} ({n_models} models x {N_CLASSES} classes)")

# Meta-learner: LogReg
meta_oof = np.zeros((len(train), N_CLASSES), dtype=np.float64)
meta_test_acc = np.zeros((len(test), N_CLASSES), dtype=np.float64)
for fi, (tr_idx, va_idx) in enumerate(folds):
    meta_lr = LogisticRegression(class_weight='balanced', C=1.0, max_iter=5000,
                                 solver='lbfgs', multi_class='multinomial', random_state=42)
    meta_lr.fit(X_meta_train[tr_idx], y[tr_idx])
    meta_oof[va_idx] = meta_lr.predict_proba(X_meta_train[va_idx])
    meta_test_acc += meta_lr.predict_proba(X_meta_test) / N_FOLDS
    fold_f1 = f1_score(y[va_idx], np.argmax(meta_oof[va_idx], axis=1), average='macro')
    print(f"Meta Fold {fi}: F1={fold_f1:.4f}")
meta_f1 = f1_score(y, np.argmax(meta_oof, axis=1), average='macro')
print(f"Meta-learner OOF: {meta_f1:.4f}")

# V12-style (usando SVCs seed 42 para compatibilidade)
oof_svc_ens = 0.25 * oof_probas['svc_A_s42'] + 0.40 * oof_probas['svc_F_s42'] + 0.35 * oof_probas['svc_F2_s42']
oof_v12 = 0.70 * oof_svc_ens + 0.30 * oof_probas['lgb1']
v12_f1 = f1_score(y, np.argmax(oof_v12, axis=1), average='macro')
print(f"V12-style OOF: {v12_f1:.4f}")

# NOVO: Dirichlet weight search (do exp15)
print("\nDirichlet weight search (20k combos)...")
best_dir_f1 = 0
best_dir_weights = None
np.random.seed(42)
for _ in range(20000):
    raw = np.random.dirichlet(np.ones(n_models))
    w = np.round(raw / 0.05) * 0.05
    if w.sum() == 0: continue
    w /= w.sum()
    blend = sum(w[i] * oof_probas[model_names[i]] for i in range(n_models))
    score = f1_score(y, blend.argmax(1), average='macro')
    if score > best_dir_f1:
        best_dir_f1, best_dir_weights = score, w.copy()
print(f"Dirichlet OOF: {best_dir_f1:.4f}")
for n, w in zip(model_names, best_dir_weights):
    if w > 0: print(f"  {n}: {w:.2f}")

oof_dirichlet = sum(best_dir_weights[i] * oof_probas[model_names[i]] for i in range(n_models))
te_dirichlet  = sum(best_dir_weights[i] * test_probas[model_names[i]] for i in range(n_models))

# Hybrid combinations
te_svc_ens = 0.25 * test_probas['svc_A_s42'] + 0.40 * test_probas['svc_F_s42'] + 0.35 * test_probas['svc_F2_s42']
te_v12 = 0.70 * te_svc_ens + 0.30 * test_probas['lgb1']

candidates = {
    'meta': (meta_f1, meta_oof, meta_test_acc),
    'v12': (v12_f1, oof_v12, te_v12),
    'dirichlet': (best_dir_f1, oof_dirichlet, te_dirichlet),
}

# Hybrids
for a, b in [('meta','v12'), ('meta','dirichlet'), ('v12','dirichlet')]:
    for alpha in [0.3, 0.5, 0.7]:
        h_name = f'{a}_{alpha:.1f}_{b}'
        h_oof = alpha * candidates[a][1] + (1-alpha) * candidates[b][1]
        h_te  = alpha * candidates[a][2] + (1-alpha) * candidates[b][2]
        h_f1 = f1_score(y, h_oof.argmax(1), average='macro')
        candidates[h_name] = (h_f1, h_oof, h_te)

best = max(candidates, key=lambda k: candidates[k][0])
final_score, oof_blend, te_blend = candidates[best]
print(f"\nBest: {best} (F1={final_score:.4f})")

In [ ]:
# =============================================================================
# Cell 6: Threshold Search + Guardrails + Submission
# =============================================================================

threshold_classes = [6, 5, 4, 3, 1, 0]
threshold_range = np.arange(0.05, 0.55, 0.01)

def apply_thresholds(proba, thresholds):
    preds = np.argmax(proba, axis=1).copy()
    for cls in threshold_classes:
        if cls in thresholds:
            mask = proba[:, cls] > thresholds[cls]
            for hc in threshold_classes:
                if hc == cls: break
                if hc in thresholds: mask = mask & (preds != hc)
            preds[mask] = cls
    return preds

baseline_f1 = f1_score(y, oof_blend.argmax(1), average='macro')
print(f'Baseline OOF: {baseline_f1:.4f}')

best_thresholds = {}
current_f1 = baseline_f1
for cls in threshold_classes:
    best_cls_f1, best_cls_thr = current_f1, None
    for thr in threshold_range:
        trial = {**best_thresholds, cls: thr}
        trial_f1 = f1_score(y, apply_thresholds(oof_blend, trial), average='macro')
        if trial_f1 > best_cls_f1:
            best_cls_f1, best_cls_thr = trial_f1, thr
    if best_cls_thr:
        best_thresholds[cls] = best_cls_thr
        current_f1 = best_cls_f1
        print(f'Class {cls}: thr={best_cls_thr:.2f}, F1={best_cls_f1:.4f}')
    else:
        print(f'Class {cls}: no improvement')

final_f1 = f1_score(y, apply_thresholds(oof_blend, best_thresholds), average='macro')
print(f'\nFinal OOF: {final_f1:.4f}')
print(classification_report(y, apply_thresholds(oof_blend, best_thresholds), digits=4))

# Submission
test_preds = apply_thresholds(te_blend, best_thresholds)
test['target'] = test_preds

def apply_clinical_guardrails(row):
    text = str(row['report']).lower()
    pred = int(row['target'])
    birads_match = re.search(r'(?:bi-?rads|categoria)\s*:?\s*(\d)', text)
    if birads_match:
        mentioned = int(birads_match.group(1))
        if 0 <= mentioned <= 6: return mentioned
    if re.search(r'resultado de cine grau 3|carcinoma|\bcdis\b|neoplasia maligna', text): return 6
    if re.search(r'n[oó]dulo\s+espiculad', text) and pred < 4: return 5
    if 'espiculad' in text and 'distor' in text and pred < 4: return 5
    if re.search(r'calcifica[cç][aã]o grosseira|calcifica[cç][aã]o vascular', text):
        if pred >= 4 and not re.search(r'espiculad|suspeit|malign|distor', text): return 2
    return pred

test['target'] = test.apply(apply_clinical_guardrails, axis=1)
submission = test[['ID', 'target']].copy()
submission['target'] = submission['target'].astype(int)
submission.to_csv('submission.csv', index=False)

print('\nPrediction distribution:')
print(submission['target'].value_counts().sort_index())
print(f'\nSubmission saved! {submission.shape}')
print(submission)